In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

os.listdir("/content/drive/MyDrive")

['Red Modern Email Signature.png',
 'Black White Simple Photo Business Email Signature.png',
 'facebook (1).png',
 'facebook.png',
 'ABDUL BASIT.png',
 'Untitled document (1).gdoc',
 'vendor-sites.gsheet',
 'Untitled document.gdoc',
 'developing knowlwdge base using INSIGHT and EXPER....gsheet',
 'Colab Notebooks',
 '.ipynb_checkpoints',
 'Classroom',
 'my_model.keras',
 'MRI_images',
 'ADNIMERGE2']

In [3]:
os.listdir("/content/drive/MyDrive/ADNIMERGE2/research_dataset")

['DXSUM_27Apr2026.csv',
 'BLCHANGE_27Apr2026.csv',
 'PTDEMOG_27Apr2026.csv',
 'UPENN_PLASMA_FUJIREBIO_QUANTERIX_27Apr2026.csv',
 'All_Subjects_APOERES_27Apr2026.csv',
 'All_Subjects_Key_MRI_27Apr2026.csv',
 'All_Subjects_Key_PET_27Apr2026.csv']

In [6]:
import pandas as pd

# 1. Start by merging Plasma and Diagnosis (using the 'DIAGNOSIS' column we found)
# We use 'inner' because we MUST have a diagnosis for every blood test.
master = pd.merge(df_plasma, df_dx[['RID', 'VISCODE', 'DIAGNOSIS']], on=['RID', 'VISCODE'], how='inner')

# 2. Add APOE data using the correct 'GENOTYPE' column
# We use 'left' join so we don't lose patients if their genetic data is missing.
master = pd.merge(master, df_apoe[['RID', 'GENOTYPE']], on='RID', how='left')

# 3. Add PET data (Keep all columns for now so we can find the Amyloid score)
master = pd.merge(master, df_pet, on=['RID', 'VISCODE'], how='left')

# 4. Filter for your "Preclinical" Target:
# VISCODE 'bl' = Baseline
# DIAGNOSIS 1 = Cognitively Normal
preclinical = master[(master['DIAGNOSIS'] == 1) & (master['VISCODE'] == 'bl')].copy()

print(f"Total Preclinical Candidates (CN at Baseline): {len(preclinical)}")

# 5. Let's find your blood protein and PET score names once and for all
print("\n--- Columns in your 'preclinical' table ---")
print(preclinical.columns.tolist())


KeyError: 'RID'

In [7]:
print("PET Columns:", df_pet.columns.tolist())
print("\nPlasma Columns:", df_plasma.columns.tolist())


PET Columns: ['image_id', 'subject_id', 'image_visit', 'image_date', 'tau_pet', 'amyloid_pet', 'radiopharmaceutical', 'pet_description']

Plasma Columns: ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'EXAMDATE', 'Primary', 'Additive', 'pT217_F', 'AB42_F', 'AB40_F', 'AB42_AB40_F', 'pT217_AB42_F', 'NfL_Q', 'GFAP_Q', 'NfL_F', 'GFAP_F', 'Comment', 'update_stamp']


In [8]:
import pandas as pd

# 1. Start with Plasma and Diagnosis
master = pd.merge(df_plasma, df_dx[['RID', 'VISCODE', 'DIAGNOSIS']], on=['RID', 'VISCODE'], how='inner')

# 2. Add APOE (Genotype)
master = pd.merge(master, df_apoe[['RID', 'GENOTYPE']], on='RID', how='left')

# 3. Add PET data (Map subject_id to PTID and image_visit to VISCODE)
# In ADNI, 'subject_id' in PET files usually matches 'PTID' in Plasma files.
master = pd.merge(master, df_pet, left_on=['PTID', 'VISCODE'], right_on=['subject_id', 'image_visit'], how='left')

# 4. Filter for your "Preclinical" Group
# DIAGNOSIS 1 = Cognitively Normal, VISCODE 'bl' = Baseline
preclinical = master[(master['DIAGNOSIS'] == 1) & (master['VISCODE'] == 'bl')].copy()

# 5. Create your "Amyloid Positive" Label (The Ground Truth)
# Usually, for amyloid_pet values, anything > 1.11 (or a Centiloid > 25) is Positive.
# Let's clean the column first to ensure it's numeric
preclinical['amyloid_pet'] = pd.to_numeric(preclinical['amyloid_pet'], errors='coerce')
preclinical['label'] = (preclinical['amyloid_pet'] > 1.11).astype(int)

print(f"Final Preclinical Dataset Size: {len(preclinical)}")
print(f"Number of Amyloid Positive cases (Label 1): {preclinical['label'].sum()}")


Final Preclinical Dataset Size: 1027
Number of Amyloid Positive cases (Label 1): 0


In [9]:
import pandas as pd

# 1. Prepare PET file: Translate visit names to 'bl'
# This is the most likely reason for the 0 result.
df_pet['image_visit'] = df_pet['image_visit'].replace({'Screening': 'bl', 'Baseline': 'bl'})

# 2. Re-run the Merge
# Link PTID (Plasma) to subject_id (PET) and VISCODE (Plasma) to image_visit (PET)
master = pd.merge(df_plasma, df_dx[['RID', 'VISCODE', 'DIAGNOSIS']], on=['RID', 'VISCODE'], how='inner')
master = pd.merge(master, df_apoe[['RID', 'GENOTYPE']], on='RID', how='left')
master = pd.merge(master, df_pet, left_on=['PTID', 'VISCODE'], right_on=['subject_id', 'image_visit'], how='left')

# 3. Filter for your target: CN at Baseline
preclinical = master[(master['DIAGNOSIS'] == 1) & (master['VISCODE'] == 'bl')].copy()

# 4. Create the Ground Truth Label
# We use pd.to_numeric first to handle any 'None' or string values
preclinical['amyloid_pet'] = pd.to_numeric(preclinical['amyloid_pet'], errors='coerce')

# Define your label (1.11 is the standard cutoff for AV45 PET)
preclinical['label'] = (preclinical['amyloid_pet'] > 1.11).astype(int)

# --- THE CHECK ---
rows_with_pet = preclinical['amyloid_pet'].notna().sum()
label_1_count = preclinical['label'].sum()

print(f"Total Rows: {len(preclinical)}")
print(f"Rows with successful PET matches: {rows_with_pet}")
print(f"Final Amyloid Positive cases (Label 1): {label_1_count}")


Total Rows: 1027
Rows with successful PET matches: 0
Final Amyloid Positive cases (Label 1): 0


In [10]:
# 1. Look at the IDs
print("First 5 PTIDs in Plasma:", df_plasma['PTID'].head().tolist())
print("First 5 subject_ids in PET:", df_pet['subject_id'].head().tolist())

# 2. Look at the Visits
print("\nUnique VISCODEs in Plasma:", df_plasma['VISCODE'].unique())
print("Unique image_visits in PET:", df_pet['image_visit'].unique())

# 3. Check for any overlap at all
common_ids = set(df_plasma['PTID']).intersection(set(df_pet['subject_id']))
print(f"\nNumber of shared Patient IDs: {len(common_ids)}")


First 5 PTIDs in Plasma: ['011_S_0002', '011_S_0008', '011_S_0008', '011_S_0010', '011_S_0010']
First 5 subject_ids in PET: ['037_S_1421', '037_S_1421', '037_S_1421', '037_S_1421', '037_S_1421']

Unique VISCODEs in Plasma: ['bl' 'v41' 'm24' 'm48' 'v31' 'm06' '4_init' 'm18' 'v11' 'y1' 'v06' 'init'
 'm60' 'v21' 'y2' 'm36' 'm72' 'm12' 'v51' 'y4' 'v03' 'v05' '4_bl' '4_sc'
 '4_m12' '4_m24']
Unique image_visits in PET: ['m12' 'm18' 'm36' 'bl' 'm06' 'v51' 'v11' 'init' 'y2' 'm24' 'v31' 'm48'
 'v06' 'v21' 'v41' 'm60' 'y1' 'y4' 'v03' 'tau' '4_init' '4_bl' '4_m24'
 '4_m12']

Number of shared Patient IDs: 1489


In [11]:
import pandas as pd

# 1. Simplify the PET file: Keep the highest amyloid_pet score per patient
# This ensures we capture the "Positive" status if they ever had it
df_pet_clean = df_pet.sort_values('amyloid_pet', ascending=False).drop_duplicates('subject_id')

# 2. Merge Plasma and Diagnosis (The "Input" features)
master = pd.merge(df_plasma, df_dx[['RID', 'VISCODE', 'DIAGNOSIS']], on=['RID', 'VISCODE'], how='inner')

# 3. Add APOE
master = pd.merge(master, df_apoe[['RID', 'GENOTYPE']], on='RID', how='left')

# 4. The "Magic Merge": Link PTID to subject_id (Ignoring visit code for the label)
# We do this because a PET scan might be 6 months after the blood test
master = pd.merge(master, df_pet_clean[['subject_id', 'amyloid_pet']], left_on='PTID', right_on='subject_id', how='left')

# 5. Filter for your target: CN at Baseline
preclinical = master[(master['DIAGNOSIS'] == 1) & (master['VISCODE'] == 'bl')].copy()

# 6. Create the Label (Ground Truth)
preclinical['amyloid_pet'] = pd.to_numeric(preclinical['amyloid_pet'], errors='coerce')

# Standard AV45 Cutoff: 1.11
preclinical['label'] = (preclinical['amyloid_pet'] > 1.11).astype(int)

# --- THE RESULTS ---
print(f"Total Rows: {len(preclinical)}")
print(f"Rows with PET data: {preclinical['amyloid_pet'].notna().sum()}")
print(f"Amyloid Positive cases (Label 1): {preclinical['label'].sum()}")
print(f"Amyloid Negative cases (Label 0): {(preclinical['label'] == 0).sum()}")


Total Rows: 541
Rows with PET data: 0
Amyloid Positive cases (Label 1): 0
Amyloid Negative cases (Label 0): 541


In [12]:
import pandas as pd

# 1. Clean the ID columns (Remove any hidden spaces and force to string)
df_plasma['PTID'] = df_plasma['PTID'].astype(str).str.strip()
df_pet['subject_id'] = df_pet['subject_id'].astype(str).str.strip()

# 2. Simplify the PET file: Take the maximum score available for each person
# We convert to numeric first to ensure 'max()' works on the scores
df_pet['amyloid_pet'] = pd.to_numeric(df_pet['amyloid_pet'], errors='coerce')
df_pet_clean = df_pet.sort_values('amyloid_pet', ascending=False).drop_duplicates('subject_id')

# 3. Merge Plasma and Diagnosis
master = pd.merge(df_plasma, df_dx[['RID', 'VISCODE', 'DIAGNOSIS']], on=['RID', 'VISCODE'], how='inner')

# 4. The "Forced" Merge on IDs
# We link PTID from Plasma directly to subject_id from PET
master = pd.merge(master, df_pet_clean[['subject_id', 'amyloid_pet']],
                  left_on='PTID', right_on='subject_id', how='left')

# 5. Filter for your target: CN at Baseline
preclinical = master[(master['DIAGNOSIS'] == 1) & (master['VISCODE'] == 'bl')].copy()

# 6. Create the Label (Ground Truth)
# We check the max value to see if we should use 1.11 (AV45) or 25 (Centiloid)
max_val = preclinical['amyloid_pet'].max()
cutoff = 1.11 if max_val < 5 else 25

preclinical['label'] = (preclinical['amyloid_pet'] > cutoff).astype(int)

# --- THE FINAL CHECK ---
print(f"Total Rows: {len(preclinical)}")
print(f"Total unique PTIDs in Plasma: {df_plasma['PTID'].nunique()}")
print(f"Total unique subject_ids in PET: {df_pet['subject_id'].nunique()}")
print(f"Rows with successful PET matches: {preclinical['amyloid_pet'].notna().sum()}")
print(f"Amyloid Positive cases (Label 1): {preclinical['label'].sum()}")


Total Rows: 541
Total unique PTIDs in Plasma: 1593
Total unique subject_ids in PET: 2361
Rows with successful PET matches: 0
Amyloid Positive cases (Label 1): 0


In [13]:
# Check File 1: DXSUM (Diagnosis)
print("--- DXSUM Sample ---")
print(df_dx[['RID', 'PTID', 'VISCODE', 'DIAGNOSIS']].dropna().head())

# Check File 2: Plasma (p-tau217)
print("\n--- Plasma Sample ---")
# Look for PTID and your p-tau column (pT217_F)
print(df_plasma[['RID', 'PTID', 'VISCODE', 'pT217_F']].dropna().head())

# Check File 3: PET (Amyloid Label)
print("\n--- PET Sample ---")
# Look for subject_id and amyloid_pet
print(df_pet[['subject_id', 'image_visit', 'amyloid_pet']].dropna().head())


--- DXSUM Sample ---
   RID        PTID VISCODE  DIAGNOSIS
0    2  011_S_0002      bl        1.0
1    3  011_S_0003      bl        3.0
2    5  011_S_0005      bl        1.0
3    8  011_S_0008      bl        1.0
4    7  022_S_0007      bl        3.0

--- Plasma Sample ---
   RID        PTID VISCODE  pT217_F
0    2  011_S_0002      bl    0.199
1    8  011_S_0008      bl    0.152
2    8  011_S_0008     v41    0.235
3   10  011_S_0010      bl    0.472
4   10  011_S_0010     m24    0.586

--- PET Sample ---
Empty DataFrame
Columns: [subject_id, image_visit, amyloid_pet]
Index: []


In [15]:
# Reload the file to ensure it isn't empty
df_pet = pd.read_csv(path + 'All_Subjects_Key_PET_27Apr2026.csv')

# Verify immediately - this should NOT be 0
print(f"Total rows in PET file after reload: {len(df_pet)}")
print(df_pet[['subject_id', 'image_visit', 'amyloid_pet']].sample(20))


Total rows in PET file after reload: 10061
       subject_id image_visit amyloid_pet
9103  036_S_10133        4_bl         NaN
7464   035_S_6722          bl           Y
2806   029_S_2395        init         NaN
5702   027_S_5277        init         NaN
581    128_S_0272          bl         NaN
2050   098_S_2079         v11         NaN
1099   010_S_0904         m18         NaN
5532   127_S_5200          y2         NaN
9554  052_S_10267        4_bl         NaN
2546   036_S_2380          y2         NaN
5941   130_S_6072          bl         NaN
9912  036_S_10812        4_bl         NaN
5999   037_S_6083          bl         NaN
8381   027_S_5083      4_init         NaN
6442   129_S_6304          bl           Y
5320   029_S_5135         v21           Y
1208   130_S_0783         m06         NaN
271    128_S_0135         m48           Y
4698   127_S_4765        init           Y
4449   007_S_4637         v21         NaN
